# A cost-efficient Deep Learning model to emulate surface mass balance (SMB) over the Antartic Peninsula

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import xarray as xr

from super_resolution import rcm_emulator_loc_entrypoint

%load_ext autoreload
%autoreload 2

Select an input date:

In [2]:
DATE = datetime(2023, 1, 1)

Select an input location:

In [3]:
LOCATION = (-75.0076, -65.1279)

Here I download the data with ddm, but API should do it internally.

Download of:
1. ERA5_MONTHLY
2. RACMO

In [ ]:
from datetime import timedelta

from download_data import download_era5_from_ddm, download_rcm_from_ddm

era5_path = download_era5_from_ddm(
    start_date=DATE,
    end_date=DATE + timedelta(days=31),
    output_dir=Path("local_data/ERA5"),
)

racmo_path = download_rcm_from_ddm(
    start_date=DATE, end_date=DATE + timedelta(days=31), output_dir=Path("local_data/RACMO")
)

### Call the entrypoint

In [ ]:
# it will take 20mins without GPU
OUTPUT_PATH = "./tmp/predictions.zarr"
rcm_emulator_loc_entrypoint.main(
    gcm_zarr_path=era5_path,
    rcm_zarr_path=racmo_path,
    location=LOCATION,
    output_zarr_path=OUTPUT_PATH,
    model_name="model_attention_unet",
)

In [43]:
preds_ds = xr.open_zarr(OUTPUT_PATH, consolidated=True)

In [ ]:
preds_ds

### Plot timeseries

In [ ]:
predictions = preds_ds["predictions"].isel(time=0).values

In [ ]:
target_tiles = ...  # TODO assign your target tiles array here
vmin = np.min([np.min(predictions), np.min(target_tiles)])
vmax = np.max([np.max(predictions), np.max(target_tiles)])

In [ ]:
y_vals = np.mean(predictions, axis=(1, 2))
y_vals_center = predictions[:, predictions.shape[1] // 2, predictions.shape[2] // 2]

era5_vars = ["d2m", "msl", "sp", "sst", "t2m", "tp", "u10", "v10"]
input_tiles = ...  # TODO assign your input tiles array here
y_precip = np.mean(input_tiles[:, 0, 5, :, :], axis=(1, 2))
y_temp = np.mean(input_tiles[:, 0, 4, :, :], axis=(1, 2))

era5_xy = ...  # TODO assign your ERA5 xarray dataset here
timeline = era5_xy.time.values

In [ ]:
my_lovely_dataset = ...  # TODO assign your xarray dataset here
my_lovely_dataset_df = my_lovely_dataset.mean(dim=["x", "y"]).to_dataframe()  # TODO - name your dataframe appropriately